<div align="center">

# <span style="color:#1976D2;">Customer Churn Prediction and Retention Strategy Using Machine Learning</span>

**Domain:** E-Commerce

**Project Type:** Supervised Machine Learning — Binary Classification

**Target Variable:** Churn

**Unit of Analysis:** Customer Account

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, and XGBoost

</div>

## <span style="color:#1976D2;">Section 1 — Business Understanding & Project Definition</span>

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 1.1. Business Problem Definition


Customer retention is a major challenge in the competitive E-Commerce industry. Losing an account can result in the loss of multiple customers and associated revenue, as each account may have several users.

The company needs a predictive model to identify accounts likely to churn and recommend targeted retention campaigns.

However, retention incentives must be financially sustainable. The challenge is therefore to predict churn accurately while recommending cost-effective strategies that protect revenue and profitability.


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 1.2. Project Objectives



**Primary Objective:** Develop a machine learning model to predict account churn and recommend financially viable retention strategies.

| Objective | Expected Outcome |
|---|---|
| Explore customer data | Understand churn patterns and account characteristics |
| Identify churn drivers | Determine factors associated with account churn |
| Develop predictive models | Predict the probability of account churn |
| Evaluate model performance | Select the most effective classification model |
| Segment at-risk accounts | Identify accounts requiring retention intervention |
| Recommend retention campaigns | Propose targeted, cost-effective offers |



<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 1.3. Dataset Overview

The dataset contains account-level information covering customer characteristics, service interactions, satisfaction, and revenue.

| Category | Variables |
|---|---|
| Identifier | AccountID |
| Target | Churn |
| Account Profile | Tenure, City_Tier, Account_user_count, account_segment |
| Demographics | Gender, Marital_Status |
| Customer Service | CC_Contacted_L12m, Service_Score, CC_Agent_Score, Complain_l12m, Day_Since_CC_connect |
| Revenue & Incentives | rev_per_month, rev_growth_yoy, coupon_used_l12m, cashback_l12m |
| Customer Preferences | Payment, Login_device |

**Target Variable:** Churn (binary classification)

**Unit of Analysis:** Account

AccountID will be excluded from model training because it is a unique identifier.




<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 1.4. Methodology

The project will follow a structured machine learning workflow.




```text
                         CHURNGUARD — PROJECT WORKFLOW

+--------------------------- DATA & MODEL DEVELOPMENT ---------------------------+
|                                                                                |
|  +---------------+   +---------------+   +---------------+   +---------------+ |
|  | 1. Data       |-->| 2. Exploratory|-->| 3. Data       |-->| 4. Model      | |
|  | Understanding |   | Analysis      |   | Preprocessing |   | Development   | |
|  +---------------+   +---------------+   +---------------+   +---------------+ |
+--------------------------------------------------------------------------|-----+
                                                                           |
                +----------------------------------------------------------+
                |
                v
+----------------------- EVALUATION & STRATEGY --------+        +-------------- STREAMLIT APPLICATION -----------------+
|                                                      |        |                                                      |
| +-------------+  +-------------+  +----------------+ |        | +--------------------------------------------------+ |
| | 5. Model    |->| 6. Churn    |->| 7. Retention   |--------> |          8. ChurnGuard Deployment|                   |
| | Evaluation  |  | Segmentation|  | Strategy & ROI | |        | +--------------------------------------------------+ |
| +-------------+  +-------------+  +----------------+ |        | Single / Batch Prediction  with Email Notification   |
|                                                      |        | Risk Scores & Recommendations                        |
|                                                      |        | CSV Export & Model Info                              |
+------------------------------------------------------+        +------------------------------------------------------+

```text
                    ChurnGuard — Repository Structure

ChurnGuard/
│
├── app.py                              # Streamlit web interface
│
├── data/
│   ├── sample.csv# Saved best traied model
│   └── Customer Churn Data.xlsx        # Original dataset
│
├── outputs/
│   ├── eda_reports.html# Saved best traied model
│   └── churn_prediction.ipynb          # EDA, modelling, evaluation & ROI
│
├── models/
│   ├── churnguard_<best_model_name>.joblib # Saved best traied model
│   └── feature_schema.json
│
├── .streamlit/secrets.toml
├── requirements.txt                    # Python dependencies
├── README.md                           # Project documentation
├── .env                                 # Environmental variables
└── .gitignore                          # Git exclusions

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 1.5. Success Criteria and Expected Deliverables


| Area | Success Criteria / Deliverable |
|---|---|
| Data Quality | Clean and validated dataset |
| Predictive Performance | Model with strong performance against baseline metrics |
| Churn Insights | Identification of key factors associated with churn |
| Risk Segmentation | Account-level churn probabilities and risk categories |
| Retention Strategy | Targeted campaigns with positive expected financial returns |
| Business Recommendations | Actionable recommendations for reducing account churn |

**Expected Outcome:** A reliable churn prediction model supported by actionable insights and financially sustainable retention recommendations.


## <span style="color:#1976D2;">Section 2: Data Understanding, EDA, & Model Development</span>


---

### 1. Data Understanding

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Environment Setup & Data Loading 

In [ ]:
# Install the required libraries

# !python -m pip install --upgrade pip
# !python -m pip install pandas numpy matplotlib seaborn scikit-learn xgboost openpyxl ipykernel jupyter

In [ ]:
# Import the necessary libraries for data analysis, visualization, and machine learning.

# Standard library imports
from pathlib import Path
import warnings
import os
import sys
from dotenv import load_dotenv
from getpass import getpass

# Data manipulation and numerical computing
import pandas as pd
import numpy as np
import math

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns
import sweetviz as sv
from mpl_toolkits.mplot3d import Axes3D
import shap
import json

# Machine learning utilities
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
from sklearn.base import clone
from sklearn.model_selection import ParameterGrid
import joblib


# Classification algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline

# Model evaluation metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, average_precision_score

# Suppress non-critical warnings for cleaner notebook output.
# Important modelling warnings should still be investigated.
warnings.filterwarnings("ignore", category=FutureWarning)

# Configure Pandas display settings.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

# Configure visualization defaults.
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Set a reproducible random seed for modelling.
RANDOM_STATE = 42

load_dotenv() 

print("All required libraries imported successfully.")


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Configure Project Directories

In [ ]:
# Locate the project root using existing project folders.
PROJECT_ROOT = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "data" / "Customer Churn Data.xlsx").is_file()
    and (p / "outputs").is_dir()
)

# Set the project root as the working directory.
os.chdir(PROJECT_ROOT)
#print("WORKING DIRECTORY:", PROJECT_ROOT)
print("WORKING DIRECTORY:", f"***/{Path.cwd().name}")


In [ ]:
# Define project directories.
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODELS_DIR = PROJECT_ROOT / "models"
DEPLOYMENT_DIR = PROJECT_ROOT / ".streamlit"

# Create directories if missing.
for path in [DATA_DIR, OUTPUT_DIR, MODELS_DIR, DEPLOYMENT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Display directory paths.
print("\nPROJECT DIRECTORIES")
for path in [PROJECT_ROOT, DATA_DIR, OUTPUT_DIR, MODELS_DIR, DEPLOYMENT_DIR]:
    print(f"***/{path.relative_to(PROJECT_ROOT.parent)}")

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Load Customer Churn Dataset

The customer churn dataset is imported from an Excel workbook using Pandas. The initial inspection confirms the dataset dimensions, available worksheets, and sample records before further analysis.


In [ ]:
# Define the Excel workbook path.
FILE_PATH = DATA_DIR / "Customer Churn Data.xlsx"

# Load the specified worksheet into a Pandas DataFrame.
df = pd.read_excel(FILE_PATH, sheet_name="Data for DSBA")

---

### 2. Exploratory Data Analysis (EDA)

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### 1. Exploratory Data Analysis (EDA) - Raw Data
Explore the data, assess its quality, and identify patterns linked to customer churn.

In [ ]:
# Standardize column names & values by stripping whitespace and converting to lowercase.
df.columns = df.columns.str.strip().str.lower()

# Set the target column and identifier column for the dataset.

TARGET_COL = "churn"
id_cols = "accountid"

# Display dataset dimensions.
print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Preview the first five records.
display(df.head())

# Examine column names, data types, and non-null counts.
df.info()

**Observation**: The dataset contains 11,260 records and 19 variables. Several columns have missing values, while some numerical features are stored as objects, indicating potential data quality issues requiring cleaning before modelling.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### 2 Data Quality Assessment

##### Check Invalid Values in Numeric Columns

In [ ]:
# Identify columns expected to contain numerical values.
numeric_cols = [
    "tenure", "account_user_count", "rev_per_month", "rev_growth_yoy", "coupon_used_for_payment",
    "day_since_cc_connect", "cashback"]

# Identify non-numeric values and summarize them in a table.
invalid_summary = []

for col in numeric_cols:
    invalid = pd.to_numeric(df[col], errors="coerce").isna() & df[col].notna()
    invalid_summary.append({
        "Column": col,
        "Invalid Count": invalid.sum(),
        "Invalid Values": df.loc[invalid, col].astype(str).unique().tolist()[:10]
    })

# Display the summary table.
display(pd.DataFrame(invalid_summary))


##### Correct Invalid Numeric Values

In [ ]:
# Create a working copy to preserve the original dataset.
df_clean = df.copy()

# Convert numerical columns; invalid values become NaN.
df_clean[numeric_cols] = df_clean[numeric_cols].apply(pd.to_numeric, errors="coerce")

# Summarize data types and missing values in a table.
summary = pd.DataFrame({
    "Data Type": df_clean[numeric_cols].dtypes.astype(str),
    "Missing Count": df_clean[numeric_cols].isna().sum(),
    "Missing (%)": (df_clean[numeric_cols].isna().mean() * 100).round(2)
})

display(summary.reset_index(names="Column"))

**Observation**: All seven columns have been successfully converted to `float64`. Missing values range from 0.03% to 7.02%, with `rev_per_month` having the highest (791 records), followed by `cashback` (473) and `Account_user_count` (444).

##### Sweetviz Report

In [ ]:
# Display dataset dimensions.
%matplotlib inline


print(f"Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Autoviz display
report = sv.analyze(df_clean, target_feat=TARGET_COL)

# Save HTML report
report.show_html("outputs/eda_report.html")

##### Data Quality Observations and Cleaning Plan

Based on the generated eda_report.html (Sweetviz EDA report), the following table summarizes the findings and recommended cleaning actions.

| QA Category     | Observation                                 | Cleaning Step          | Rationale                 |
| --------------- | ------------------------------------------- | ---------------------- | ------------------------- |
| Missing Values  | `rev_per_month`: 7.02%                      | Median imputation      | Handles skewness          |
|                 | `cashback`: 4.20%                           | Median imputation      | Reduces outlier impact    |
|                 | `account_user_count`: 3.94%                 | Mode imputation        | Preserves typical count   |
|                 | `day_Since_CC_connect`: 3.18%               | Median imputation      | Retains records           |
|                 | `complain_ly`: 3.17%                        | Mode imputation        | Preserves common category |
|                 | `tenure`: 1.94%                             | Median imputation      | Retains records           |
|                 | Other missing values: under 2%              | Type-based imputation  | Preserves data            |
| Duplicates      | No duplicate rows                           | No action              | Data is unique            |
|                 | `accountid`: 11,260 unique                  | Exclude from modelling | Identifier only           |
| Incorrect Labels | `gender`: 4 categories (`M`, `F`, `Male`, `Female`) | Convert `M` → `Male`, `F` → `Female` | Standardize to 2 categories |
|                  | `login_device`: 3 categories, including `&&&&`      | Replace `&&&&` → `Others`            | Correct invalid label       |
| Other QA Checks | Extreme tenure, revenue and cashback values | Investigate outliers   | Validates unusual values  |
|                 | `churn`: Binary, no missing values          | Check class balance    | Supports model evaluation |

Note: Fit imputation on training data only to prevent data leakage.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### 3 Univariate Analysis — Distribution and Outlier Assessment

In [ ]:
# EDA: Identify numeric and categorical columns

# Standardize numerical values by stripping whitespace and converting to lowercase.
df_clean = df_clean.apply(lambda col: col.str.strip().str.lower() if col.dtype == "object" else col)

df_eda = df_clean.copy()

features = [c for c in df_eda.columns if c not in [id_cols, TARGET_COL]]

X_eda = df_eda[features].copy()
num_var = X_eda.select_dtypes(include=[np.number]).columns.tolist()
cat_var = [c for c in X_eda.columns if c not in num_var]

print("ID cols:", id_cols)
print("Numeric cols:", num_var)
print("Categorical cols:", cat_var)

##### 3.1 Target Distribution (Univariate)

In [ ]:
# Plot target class distribution (%).
cs_pct = df_eda[TARGET_COL].value_counts(normalize=True).sort_index() * 100

ax = cs_pct.plot.bar(figsize=(5, 4), rot=0)
ax.bar_label(ax.containers[0], fmt="%.1f%%", padding=3)

ax.set(title=f"{TARGET_COL.title()} Distribution",
       xlabel=TARGET_COL.replace("_", " ").title(),
       ylabel="Percentage (%)",
       ylim=(0, 110))

plt.tight_layout()
plt.show()

The dataset is imbalanced, with 83.2% non-churned customers and 16.8% churned customers, requiring stratified sampling and appropriate evaluation metrics.

##### 3.2 Numeric Distribution (Univariate)

In [ ]:
# Numeric Distribution — All Variables
fig, ax = plt.subplots(len(num_var), 3, figsize=(15, 3.5*len(num_var)),
                       gridspec_kw={"width_ratios": [2, 2, 1]}, squeeze=False)

for i, c in enumerate(num_var):
    s = df_eda[c].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    lf, uf = q1 - 1.5*(q3-q1), q3 + 1.5*(q3-q1)
    outliers = ((s < lf) | (s > uf)).sum()

    # Histogram
    sns.histplot(s, kde=True, ax=ax[i, 0])
    for v, color in [(s.mean(), "red"), (s.median(), "black"), (lf, "gray"), (uf, "gray")]:
        ax[i, 0].axvline(v, color=color, ls="--")
    ax[i, 0].set_title(f"{c} — Distribution")

    # Boxplot
    sns.boxplot(x=s, ax=ax[i, 1])
    ax[i, 1].set_title(f"{c} — Boxplot")

    # Outlier summary
    summary = [["Min", f"{s.min():.2f}"], ["Max", f"{s.max():.2f}"],
               ["Lower IQR", f"{lf:.2f}"], ["Upper IQR", f"{uf:.2f}"],
               ["Outliers", outliers], ["Outlier %", f"{outliers/len(s)*100:.2f}%"]]
    ax[i, 2].axis("off")
    ax[i, 2].table(cellText=summary, loc="center", colWidths=[0.55, 0.45]).scale(1, 1.3)
    ax[i, 2].set_title("Outlier Summary")

plt.tight_layout()
plt.show()


**Outlier Assessment and Treatment**

| Variable                  | Issue Type                                 | Severity | Action Taken                                         |
| ------------------------- | ------------------------------------------ | -------- | ---------------------------------------------------- |
| `tenure`                  | Right skew + extreme values (1.26%)        | Medium   | Retain; validate extremes                            |
| `cc_contacted_ly`         | Right skew (0.38%)                         | Low      | Retain                                               |
| `account_user_count`      | Discrete values flagged by IQR (7.04%)     | Low      | Retain                                               |
| `rev_per_month`           | Strong right skew (1.77%)                  | Medium   | Retain; consider log transform                       |
| `coupon_used_for_payment` | Right skew + outliers (12.26%)             | High     | Retain; consider log transform                       |
| `day_since_cc_connect`    | Right skew (0.30%)                         | Low      | Retain                                               |
| `cashback`                | Strong right skew + extreme values (8.15%) | High     | Retain; validate extremes and consider log transform |

Outliers will be retained because they may reflect genuine customer behaviour and are generally well handled by tree-based models. Skewed variables may be transformed where necessary rather than removing observations.

Note: Transformations are proposed, not yet applied.

##### 3.3 Categorical Distribution (Univariate)

In [ ]:
# Categorical Distribution (Univariate)

ncols = 2
nrows = math.ceil(len(cat_var) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
axes = np.array(axes).reshape(-1)

for ax, c in zip(axes, cat_var):
    pct = df_eda[c].value_counts(normalize=True, dropna=False) * 100

    sns.barplot(x=pct.values, y=pct.index.astype(str), ax=ax)
    ax.set(title=f"{c} Distribution", xlabel="Percentage (%)", ylabel="", xlim=(0, 110))

    for i, v in enumerate(pct):
        ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=9)

# Remove unused subplots.
for ax in axes[len(cat_var):]:
    ax.remove()

plt.tight_layout()
plt.show()


**Summary of Categorical Variables — Univariate Analysis**

| Variable          | Key Observation                                                      | Business Insight                                                        |
| ----------------- | -------------------------------------------------------------------- | ----------------------------------------------------------------------- |
| `payment`         | Debit cards (40.7%) and credit cards (31.2%) dominate; 1.0% missing. | Card payments dominate; missing values require imputation.              |
| `gender`          | Male (56.2%), female (37.1%); 1.0% missing and inconsistent labels.  | Standardize labels and impute missing values.                           |
| `account_segment` | Super (36.1%) and Regular Plus (34.3%) dominate; 0.9% missing.       | Higher-tier segments dominate; inconsistent labels require correction.  |
| `marital_status`  | Married customers (52.0%) dominate; 1.9% missing.                    | Married customers form the largest group; imputation required.          |
| `login_device`    | Mobile (66.4%) dominates; 2.0% missing and invalid labels.           | Mobile is the primary access channel; clean invalid and missing values. |

Overall, categorical variables show low missingness (0.9%–2.0%), with mobile usage, card payments, and higher-tier account segments dominating. Missing values will be imputed using the mode on training data, while inconsistent labels will be standardized before modelling.

##### 3.4 Overall Univariate Summary

- Customer churn is imbalanced, with 83.2% retained and 16.8% churned customers.
- Tenure, revenue, cashback, and coupon usage exhibit right-skewed distributions and outliers.
- Debit and credit cards dominate payment methods, accounting for 71.9% of customers.
- Super and Regular Plus are the largest account segments, representing 70.4% of customers.
- Mobile devices dominate customer access (66.4%).
- Missing values and inconsistent categorical labels require preprocessing before modelling.

**Strategic Implication**

The customer base is predominantly mobile-oriented, card-paying, and concentrated in higher-tier account segments. The imbalanced churn distribution requires stratified sampling and appropriate evaluation metrics, while skewed numerical variables and data quality issues should be addressed during preprocessing to support reliable churn prediction.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### 4 Bivariate Analysis — Categorical Variables vs Target

##### 4.1 Categorical Distribution (Bivariate)

In [ ]:
# Bivariate Analysis — Categorical Variables vs Target
ncols = 2
nrows = (len(cat_var) + 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4*nrows))
axes = np.array(axes).reshape(-1)

for ax, c in zip(axes, cat_var):
    # Include missing values as a separate category.
    prop = pd.crosstab(df_eda[c].fillna("Missing"), df_eda[TARGET_COL], normalize="index")
    prop = prop.sort_values(by=1, ascending=False)

    prop.plot.bar(stacked=True, ax=ax, colormap="Set2", rot=0)
    ax.set(title=f"{c} vs {TARGET_COL}", xlabel="", ylabel="Percentage (%)", ylim=(0, 1))
    ax.set_yticks(np.arange(0, 1.1, 0.2), labels=[f"{v:.0%}" for v in np.arange(0, 1.1, 0.2)])
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title=TARGET_COL)

    # Annotate percentages within bars.
    for container in ax.containers:
        ax.bar_label(container, fmt=lambda v: f"{v:.1%}" if v >= 0.05 else "",
                     label_type="center", fontsize=8)

# Remove unused subplots.
for ax in axes[len(cat_var):]:
    ax.remove()

plt.tight_layout()
plt.show()


**Categorical Variables vs. Customer Churn**

| Variable | Key Observation | Impact | Interpretation |
|---|---|---|---|
| `payment` | Cash highest (25.0%); credit card lowest (14.2%). | Moderate | Churn varies by payment method. |
| `gender` | Rates range from 14.1% to 20.7%. | Weak–Moderate | Standardize labels before comparing. |
| `account_segment` | Regular Plus highest (27.4%); Super Plus lowest (~2.1%). | Strong | Rates differ substantially by segment. |
| `marital_status` | Single highest (26.9%); married lowest (11.5%). | Strong | Single customers show higher churn. |
| `login_device` | Computer users: 19.8%; mobile users: 15.7%. | Weak | Rates differ slightly by device. |

Account segment and marital status show the largest differences. Missing and inconsistent categories need cleanup. These are associations, not proof of causation.

##### 4.2 Numerical Distribution (Bivariate)

In [ ]:
# Numeric Bivariate Analysis — All Variables vs Target
fig, axes = plt.subplots(len(num_var), 3, figsize=(15, 3.5*len(num_var)), squeeze=False)

for i, c in enumerate(num_var):
    # Density by churn status.
    sns.kdeplot(data=df_eda, x=c, hue=TARGET_COL, ax=axes[i, 0], common_norm=False)
    axes[i, 0].set_title(f"{c} — Density by {TARGET_COL}")

    # Boxplots with and without outliers.
    med = df_eda.groupby(TARGET_COL)[c].median()
    for j, show in enumerate([True, False], start=1):
        ax = axes[i, j]
        sns.boxplot(data=df_eda, x=TARGET_COL, y=c, showfliers=show, ax=ax)
        ax.set_title(f"{c} — {'With' if show else 'Without'} Outliers")

        # Annotate median values.
        for k, v in enumerate(med):
            ax.text(k, v, f"{v:.1f}", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()


**Numeric Variables vs. Customer Churn**

| Variable | Retained Median | Churned Median | Pattern | Impact | Interpretation |
|---|---:|---:|---|---|---|
| `tenure` | 10 | 1 | Strong separation | Strong | New customers churn more. |
| `city_tier` | 1 | 1 | Nearly identical | Weak | Little difference. |
| `cc_contacted_ly` | 16 | 17 | Overlap | Weak | Contacts may reflect dissatisfaction. |
| `service_score` | 3 | 3 | Similar | Weak | Little difference. |
| `account_user_count` | 3 | 4 | Noticeable shift | Moderate | Larger accounts churn more. |
| `cc_agent_score` | 3 | 4 | Noticeable shift | Moderate | Scores differ by churn status. |
| `rev_per_month` | 5 | 5 | Overlap; right-skewed | Weak | Revenue poorly separates churn. |
| `complain_ly` | 0 | 1 | Strong separation | Strong | Complaints relate to higher churn. |
| `rev_growth_yoy` | 15 | 15 | Nearly identical | Weak | Little difference. |
| `coupon_used_for_payment` | 1 | 1 | Overlap; right-skewed | Weak | Little separation. |
| `day_since_cc_connect` | 4 | 2 | Noticeable shift | Moderate | Recent contact relates to churn. |
| `cashback` | ~168 | ~152 | Overlap; right-skewed | Moderate | Lower cashback relates to churn. |

Tenure and complaints show the clearest differences; impact levels are descriptive, not tests of significance.

##### 4.3 Overall Bivariate Summary

- Tenure and complaint history show the strongest associations with customer churn.
- Account segment and marital status reveal significant differences in observed churn rates.
- Payment method influences churn patterns, with cash-on-delivery customers showing higher churn.
- Account user count, agent score, contact recency, and cashback show moderate associations with churn.
- Gender and login device show relatively weak differences.
- Revenue, service score, city tier, and coupon usage provide limited standalone separation.

**Strategic Implication**

Customer churn is primarily associated with customer tenure, complaint history, account segmentation, and engagement patterns. Retention strategies should prioritize new customers, customers with complaints, and high-churn account segments, while multivariate modelling will help validate the most important predictors.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### 5 Multivariate Analysis (Feature–Feature Relationships)

- Assesses predictor relationships to uncover redundancy and interaction effects.

##### 5.1 Numerical Correlation

In [ ]:
# Multivariate Analysis — Numerical Correlation
corr = df_eda[num_var + [TARGET_COL]].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Numerical Variables — Correlation Matrix")
plt.tight_layout()
plt.show()


**Correlation Analysis**

| Variable | Correlation | Impact | Summary |
|---|---:|---|---|
| `complain_ly` | +0.25 | Moderate | Complaints relate to higher churn. |
| `tenure` | -0.23 | Moderate | Longer tenure relates to lower churn. |
| `day_since_cc_connect` | -0.15 | Weak | Recent contact relates to higher churn. |
| `account_user_count` | +0.11 | Weak | Larger accounts show slightly higher churn. |
| `cc_agent_score` | +0.11 | Weak | Slight positive association. |
| `city_tier` | +0.08 | Weak | Little linear association. |
| Other numeric variables | -0.03 to +0.07 | Negligible | Little linear association. |

Complaints and tenure have the strongest correlations with churn. Predictor correlations are generally low; weak correlations do not exclude nonlinear effects or interactions.

##### 5.2 Numerical Analysis — Pairplot

In [ ]:
# To Observe class separation, clustering, and possible interactions.
top_vars = df_eda[num_var].corrwith(df_eda[TARGET_COL]).abs().nlargest(5).index.tolist() # select top 5 correlated numerical features with target

sns.pairplot(
    df_eda[top_vars + [TARGET_COL]].dropna(),
    hue=TARGET_COL,
    diag_kind="kde",
    corner=True
    ,plot_kws={"alpha": 0.3, "s": 12} # adjust point transparency and size for better visibility
)
plt.show()


**Pairplot Observation**

| Key Observation                                           | Impact   | Interpretation                                          |
| --------------------------------------------------------- | -------- | ------------------------------------------------------- |
| Churned customers cluster at shorter tenure periods.      | Strong   | New customers face higher churn risk.                   |
 | Complaint history shows noticeable class separation.      | Strong   | Complaints are associated with churn.                   |
| Contact recency and account size show partial separation. | Moderate | Customer engagement may contribute to churn prediction. |
 | Agent scores show substantial class overlap.              | Weak     | Limited standalone differentiation.                     |


##### 5.3 Categorical Interaction - 3D Cube Plot

In [ ]:
# Multivariate Analysis — 3D Cube Plot

cols = ["tenure", "cc_contacted_ly", "account_user_count"] # These
plot_df = df_eda[cols + [TARGET_COL]].dropna()

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    plot_df[cols[0]], plot_df[cols[1]], plot_df[cols[2]],
    c=plot_df[TARGET_COL], cmap="coolwarm", alpha=0.35, s=10
)

ax.set(xlabel=cols[0], ylabel=cols[1], zlabel=cols[2],
       title="Customer Churn — 3D Feature Distribution")

fig.colorbar(scatter, ax=ax, label=TARGET_COL, ticks=[0, 1], shrink=0.65)
plt.tight_layout()
plt.show()


**3D Plot Observation**

 These three variables were selected to explore the combined effects of customer lifecycle, contact frequency, and account size, rather than simply choosing the three strongest individual predictors.

| Key Observation                                           | Impact   | Interpretation                                          |
 | --------------------------------------------------------- | -------- | ------------------------------------------------------- |
| Churned customers concentrate at shorter tenure.          | Strong   | Early customer lifecycle is associated with churn.      |
 | Contact frequency varies across both churn classes.       | Weak     | Contact frequency alone provides limited separation.    |
| Account sizes overlap considerably across churn classes.  | Weak     | No distinct 3D churn clusters are evident.              |

##### 5.4 Overall Multivariate Summary

- Complaint history (+0.25) and tenure (-0.23) show the strongest correlations with churn.
- Account size, agent score, and contact recency demonstrate weaker but potentially useful associations.
- Low correlations among predictors suggest limited multicollinearity.
- Pairplot analysis confirms clearer churn separation for tenure and complaints, with substantial overlap across other variables.
- 3D analysis shows churn concentration among newer customers but no distinct clusters across tenure, contact frequency, and account size.
- Combined predictors may improve churn classification beyond individual variables.

**Strategic Implication**

Customer churn is primarily associated with customer lifecycle and complaint history, supported by engagement and account characteristics. Retention strategies should prioritize new customers and those reporting complaints, while machine learning models should evaluate combined feature effects to identify high-risk customers more accurately.

---
### 3. Data Preprocessing

EDA findings guide preprocessing to improve data quality while preserving useful information.

| Finding | Action | Purpose |
|---|---|---|
| Missing values | Impute with median/mode | Retain records |
| Inconsistent labels | Standardize categories | Avoid duplicates |
| `accountid` is an identifier | Exclude from model features; retain for customer segmentation | Avoid identifier-based predictions while enabling segment-level analysis |
| Outliers | Retain initially | Preserve valid variation |
| Skewed data | Transform if needed | Improve model fit |
| Categorical features | One-hot encode | Enable model training |
| Different scales | Standardize | Support scale-sensitive models |
| Class imbalance | Stratify train/test split | Preserve class proportions |
| Data leakage risk | Fit preprocessing on training data only | Ensure fair evaluation |

Use `df_clean` with corrected numerical values; keep the original dataset unchanged.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Finalize the Cleaned Dataset

The categorical plots revealed additional inconsistent labels in `account_segment`. We should correct these alongside `gender` and `login_device` before splitting the dataset.

In [ ]:
# Finalize categorical cleaning.
df_model = df_clean.copy()
cat_cols = df_model.select_dtypes(include=["object", "string"]).columns
df_model[cat_cols] = df_model[cat_cols].apply(lambda s: s.str.strip().str.lower())

# Correct inconsistent labels.
fixes = {
    "gender": {"m": "male", "f": "female"},
    "login_device": {"&&&&": "others"},
    "account_segment": {"regular +": "regular plus", "super +": "super plus"}
}
df_model = df_model.replace(fixes)

# Verify cleaned categories in one table.
cols = list(fixes)
summary = pd.concat({c: df_model[c].value_counts(dropna=False) for c in cols},
                    names=["Variable", "Category"]).reset_index(name="Count")
display(summary)

In [ ]:
# Post-Preprocessing Validation: Unique Categorical Values and Numerical Ranges
# Categorical features: unique values
for col in df_model.select_dtypes(exclude="number"):
    print(f"{col}: {sorted(df_model[col].dropna().astype(str).unique())}")

# Numerical features: minimum and maximum values
display(df_model.select_dtypes(include="number").agg(["min","max"]).T)

**Observation**: Following preprocessing, the dataset contains five categorical predictors with defined categories and numerical variables within their observed ranges, providing a consistent foundation for model development.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Define Features and Target

Based on EDA, we retain all predictors initially, excluding only the account identifier. Feature selection can be revisited after model evaluation.

In [ ]:
# Define predictors and target.
TARGET_COL # defined earlier

X = df_model.drop(columns=[TARGET_COL, "accountid"])
y = df_model[TARGET_COL]

# Identify numerical and categorical predictors.
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print(f"Features: {X.shape[1]}")
print(f"Numerical: {len(num_cols)}")
print(f"Categorical: {len(cat_cols)}")
print(f"Target distribution:\n{y.value_counts(normalize=True).mul(100).round(1)}")

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Stratified train/test split

We will use an 80:20 split, preserving the observed churn distribution in both datasets.

In [ ]:
# Split before fitting imputation, encoding or scaling.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Verify split and churn proportions.
split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing"],
    "Records": [len(y_train), len(y_test)],
    "Churn (%)": [y_train.mean()*100, y_test.mean()*100]
}).round(2)

display(split_summary)


In [ ]:
# Retain account IDs for campaign targeting.
train_accounts = df_model.loc[X_train.index, id_cols].copy()
test_accounts = df_model.loc[X_test.index, id_cols].copy()

print(f"Training accounts: {len(train_accounts):,}")
print(f"Testing accounts: {len(test_accounts):,}")

**Account ID Preservation**

| Component | Action | Purpose |
|---|---|---|
| `accountid` | Excluded from model features | Prevent identifier-based predictions |
| Train/test IDs | Preserved separately and aligned by index | Match predictions to accounts |
| Account-level predictions | Linked to retained IDs | Support targeted retention |

IDs are excluded from modeling but retained for account segmentation and outreach.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Build preprocessing pipelines

The numerical pipeline uses median imputation and scaling. For discrete variables such as `account_user_count` and `complain_ly`, we will use mode imputation, as specified in the cleaning plan.

In [ ]:
# Separate discrete and continuous numerical variables.
discrete_cols = [c for c in ["account_user_count", "complain_ly"] if c in num_cols]
continuous_cols = [c for c in num_cols if c not in discrete_cols]

# Preprocessing pipelines fitted only on training data.
preprocessor = ColumnTransformer([
    ("continuous", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), continuous_cols),

    ("discrete", SimpleImputer(strategy="most_frequent"), discrete_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

print("Preprocessing pipeline configured successfully.")
print(f"Continuous: {len(continuous_cols)} | Discrete: {len(discrete_cols)} | Categorical: {len(cat_cols)}")


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Preprocessing Observations

| Component | Observation | Implication |
|---|---|---|
| Training set | 9,008 records (80%) | Used to train models |
| Test set | 2,252 records (20%) | Held out for evaluation |
| Churn rate | 16.84% train; 16.83% test | Stratification preserved class balance |
| Features | 10 continuous, 2 discrete, 5 categorical | Preprocessing configured for each type |
| Missing data | Imputation in pipeline | Statistics learned from training data |
| Data leakage | Pipeline not yet fitted | Test data remains isolated |

The data is ready for modeling, with preprocessing contained in each model pipeline.

---
### 4. Model Development

Five models were chosen to predict account churn, from an interpretable baseline to advanced ensemble methods.

| Model | Purpose |
|---|---|
| Logistic Regression | Interpretable baseline |
| Decision Tree | Detect nonlinear patterns |
| Random Forest | Improve stability |
| Gradient Boosting | Capture complex patterns |
| XGBoost | Optimize boosting performance |

Use consistent preprocessing and control overfitting. The goal is to identify high-risk accounts while limiting unnecessary retention costs.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Helper Function

In [ ]:
# Train models using identical preprocessing.
def train_models(models, preprocessor, X_train, y_train):
    """Fit each model with a fresh copy of the shared preprocessing pipeline.
       Displays training progress, and returns a dictionary of trained models for evaluation and comparison.
    """
    trained = {}
    for name, model in models.items():
        # Keep preprocessing and classifier instances independent for each model.
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(model)),
        ])
        trained[name] = pipeline.fit(X_train, y_train)
        print(f"{name}: Training completed")
    return trained

In [ ]:
# Evaluate trained classification models.

def evaluate_models(trained_models, X_train, X_test, y_train, y_test, show_cm=None):
    """Evaluate models on train/test data, rank by test PR-AUC, and optionally plot confusion matrices."""
    results, matrices = [], {}
    datasets = [("Train", X_train, y_train), ("Test", X_test, y_test)]
    show_cm = len(trained_models) == 1 if show_cm is None else show_cm

    # Display Evaluation Metrics
    for name, model in trained_models.items():
        for dataset, X, y in datasets:
            pred, prob = model.predict(X), model.predict_proba(X)[:, 1]
            results.append({
                "Model": name, "Dataset": dataset,
                "Accuracy": accuracy_score(y, pred),
                "Precision": precision_score(y, pred, zero_division=0),
                "Recall": recall_score(y, pred, zero_division=0),
                "F1-Score": f1_score(y, pred, zero_division=0),
                "ROC-AUC": roc_auc_score(y, prob),
                "PR-AUC": average_precision_score(y, prob),
            })
            if show_cm:
                matrices[name, dataset] = confusion_matrix(y, pred, labels=[0, 1])

    results_df = pd.DataFrame(results)

    # Use held-out test PR-AUC to rank models.
    ranking = (results_df.query("Dataset == 'Test'")
               .sort_values("PR-AUC", ascending=False)["Model"].tolist())
    results_df["Model"] = pd.Categorical(results_df["Model"], ranking, ordered=True)
    results_df["Dataset"] = pd.Categorical(results_df["Dataset"], ["Train", "Test"], ordered=True)
    results_df = results_df.sort_values(["Model", "Dataset"]).reset_index(drop=True)

    if show_cm:
        import math

        ncols = 4 if len(ranking) > 1 else 2
        fig, axes = plt.subplots(
            math.ceil(2 * len(ranking) / ncols), ncols,
            figsize=(4 * ncols, 3.5 * math.ceil(2 * len(ranking) / ncols)),
            squeeze=False,
        )
        axes = axes.ravel()

        # Plot Confusion Matrix
        for i, name in enumerate(ranking):
            for j, dataset in enumerate(("Train", "Test")):
                ax = axes[2 * i + j]
                ConfusionMatrixDisplay(
                    matrices[name, dataset], display_labels=["Retained", "Churned"]
                ).plot(ax=ax, cmap="Blues", colorbar=False)
                ax.set_title(f"{name} - {dataset}")

        for ax in axes[2 * len(ranking):]:
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    return results_df

In [ ]:
# Tune each model with stratified cross-validation and rank by validation PR-AUC.

def tune_models(models, tuning_params, preprocessor, X_train, y_train):
    """Return the best fitted model and CV summary for each classifier."""
    # Preserve churn proportions in each fold.
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    tuned_models, results = {}, []

    for name, model in models.items():
        print(f"\nTuning {name}...")

        # Clone preprocessing and classifier so each search is independent.
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(model)),
        ])

        # Search parameter combinations using average precision (PR-AUC).
        search = RandomizedSearchCV(
            pipeline,
            tuning_params[name],
            n_iter=min(20, len(ParameterGrid(tuning_params[name]))),
            scoring="average_precision",
            cv=cv,
            n_jobs=-1,
            random_state=42,
            verbose=1,
            refit=True,
        )
        search.fit(X_train, y_train)
        tuned_models[name] = search.best_estimator_
        results.append({
            "Model": name,
            "CV PR-AUC": search.best_score_,
            "Best Parameters": search.best_params_,
        })

        print(f"Best parameters: {search.best_params_}")
        print(f"Best CV PR-AUC: {search.best_score_:.3f}")

    # Summarize models from highest to lowest cross-validation PR-AUC.
    summary = (
        pd.DataFrame(results)
        .sort_values("CV PR-AUC", ascending=False)
        .set_index("Model")
    )
    display(summary.round(3))
    return tuned_models, summary

In [ ]:
# Plot confusion matrix only

def plot_confusion_matrices(trained_models, tuned_models, X_test, y_test, ranking=None):
    """Plot baseline and tuned test confusion matrices in model-ranking order."""
    names = list(ranking) if ranking is not None else list(tuned_models)
    ncols = 4
    nrows = (2 * len(names) + ncols - 1) // ncols

    # Create two plots per model: baseline and tuned.
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3.5 * nrows), squeeze=False)
    axes = axes.ravel()

    for i, name in enumerate(names):
        for j, (label, models) in enumerate((("Baseline", trained_models), ("Tuned", tuned_models))):
            cm = confusion_matrix(y_test, models[name].predict(X_test), labels=[0, 1])
            ConfusionMatrixDisplay(cm, display_labels=["Retained", "Churned"]).plot(
                ax=axes[2 * i + j], cmap="Blues", colorbar=False
            )
            axes[2 * i + j].set_title(f"{name} — {label}")

    # Hide unused subplot slots.
    for ax in axes[2 * len(names):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Compare models with stratified CV and plot out-of-fold confusion matrices.

def evaluate_cv(models, preprocessor, X_train, y_train):
    """Rank models by 5-fold PR-AUC and show their out-of-fold confusion matrices."""
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = {
        "Precision": "precision", "Recall": "recall", "F1": "f1",
        "ROC-AUC": "roc_auc", "PR-AUC": "average_precision",
    }
    results, matrices = [], {}

    for name, model in models.items():
        # Use existing pipelines as-is; otherwise add shared preprocessing.
        pipeline = clone(model) if isinstance(model, Pipeline) else Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(model)),
        ])

        scores = cross_validate(pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
        # Out-of-fold predictions keep each row unseen during its prediction.
        pred = cross_val_predict(pipeline, X_train, y_train, cv=cv, n_jobs=-1)
        results.append({
            "Model": name,
            **{metric: scores[f"test_{metric}"].mean() for metric in scoring},
            "PR-AUC SD": scores["test_PR-AUC"].std(),
        })
        matrices[name] = confusion_matrix(y_train, pred, labels=[0, 1])

    # Sort models by mean cross-validation PR-AUC.
    cv_df = pd.DataFrame(results).sort_values("PR-AUC", ascending=False).set_index("Model")
    display(cv_df.T.round(4))

    fig, axes = plt.subplots(1, len(cv_df), figsize=(4 * len(cv_df), 4), squeeze=False)
    for ax, name in zip(axes.ravel(), cv_df.index):
        ConfusionMatrixDisplay(
            matrices[name], display_labels=["Retained", "Churned"]
        ).plot(ax=ax, cmap="Blues", colorbar=False)
        ax.set_title(name)
    plt.tight_layout()
    plt.show()
    return cv_df

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Configure & Train Model (Baseline)
With churn at ~16.8%, we’ll address class imbalance where supported and apply consistent preprocessing, fitted on training data only.

In [ ]:
# Calculate class imbalance ratio.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Configure baseline classification models.
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, scale_pos_weight=scale_pos_weight,
                             eval_metric="logloss", random_state=42)
}

In [ ]:
# Train Baseline Models
trained_models = train_models(models, preprocessor, X_train, y_train)

# Summarize trained models.
model_summary = pd.DataFrame({
    "Model": list(trained_models),
    "Training Records": len(X_train),
    "Features": X_train.shape[1],
    "Status": "Trained"
})

display(model_summary)

**Model Development**

| Component | Action | Purpose |
|---|---|---|
| Models | Compare five algorithms | Assess different approaches |
| Class imbalance | Apply class weights where supported | Better account for churn cases |
| Preprocessing | Include in each pipeline | Prevent data leakage |
| Training data | Use the same 9,008 records | Ensure fair comparison |
| Features | Start with all 17 predictors | Assess their combined value |
| `accountid` | Keep separate from features | Support campaign targeting |

**Note:** Models will be compared to identify at-risk customers while minimizing unnecessary retention costs. Class weighting may affect probability calibration, so assess calibration before using predictions to guide campaign spending.

## <span style="color:#1976D2;">Section 3: Evaluation & Deployment</span>

Following the training of Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, and XGBoost, we will evaluate their ability to identify accounts at risk of churn.

Since EDA revealed a 16.8% churn rate, accuracy alone is insufficient. We will prioritize Recall, F1-score, and PR-AUC, while monitoring Precision to avoid unnecessary retention campaign expenditure.

---
### 5. Model Evaluation


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Evaluate Baseline Models

In [ ]:
# Evaluate model performance on training and testing data.
baseline_results = evaluate_models(trained_models, X_train, X_test, y_train, y_test)
display(baseline_results)

**Observation:** Random Forest and XGBoost perform best on test data, but perfect training scores suggest possible overfitting. Tune tree complexity and regularization, then re-evaluate with cross-validation. XGBoost currently offers the strongest balance of precision and recall.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Overfitting Mitigation Through Model Tuning
The initial evaluation revealed overfitting in Decision Tree, Random Forest, and XGBoost. To improve generalization, model complexity will be controlled through depth restrictions, minimum leaf sizes, learning rates, and regularization.

##### Hyperparameter Tuning for All Five Models

In [ ]:
# Hyperparameter search spaces for all five models.
tuning_params = {
    "Logistic Regression": {
        "classifier__C": [0.001, 0.01, 0.1, 1, 10],
        "classifier__solver": ["lbfgs", "liblinear"]
    },
    "Decision Tree": {
        "classifier__max_depth": [3, 5, 7, 10, 15],
        "classifier__min_samples_split": [10, 20, 30, 50],
        "classifier__min_samples_leaf": [5, 10, 15, 20],
        "classifier__max_features": [None, "sqrt", "log2"]
    },
    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [5, 7, 10, 12],
        "classifier__min_samples_split": [10, 20, 30],
        "classifier__min_samples_leaf": [5, 10, 15, 20],
        "classifier__max_features": ["sqrt", "log2"],
        "classifier__max_samples": [0.7, 0.8, 0.9]
    },
    "Gradient Boosting": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__learning_rate": [0.01, 0.05, 0.1, 0.2],
        "classifier__max_depth": [2, 3, 4, 5],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__min_samples_leaf": [1, 2, 4],
        "classifier__subsample": [0.8, 0.9, 1.0]
    },
    "XGBoost": {
        "classifier__max_depth": [3, 4, 5, 6],
        "classifier__min_child_weight": [1, 3, 5],
        "classifier__gamma": [0, 0.1, 0.2],
        "classifier__subsample": [0.7, 0.8, 0.9, 1.0],
        "classifier__colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "classifier__colsample_bylevel": [0.7, 0.8, 0.9, 1.0],
        "classifier__learning_rate": [0.01, 0.03, 0.05, 0.1],
        "classifier__n_estimators": [200, 300, 500],
        "classifier__reg_lambda": [0.4, 0.5, 0.6]
    }
}

In [ ]:
# Train tuned Models
tuned_models, tuning_results = tune_models(
    models, tuning_params, preprocessor, X_train, y_train
)

In [ ]:
# Evaluate Tuned  Models
tuned_results = evaluate_models(  tuned_models, X_train, X_test, y_train, y_test )
display(tuned_results)

**Observation:** 
Tuned XGBoost achieved the highest test recall (94%) and PR-AUC (0.96), while Gradient Boosting achieved higher precision (93%) but lower recall (84%). These results indicate that XGBoost is particularly effective at identifying customers likely to churn. However, final model selection is based on cross-validation PR-AUC to assess consistency across training-data splits, rather than relying solely on test-set performance.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Compare Baseline vs Tuned Models

In [ ]:
# Compare Baseline vs Tuned Models (Test Results)
baseline = baseline_results.query("Dataset == 'Test'").set_index("Model")
tuned = tuned_results.query("Dataset == 'Test'").set_index("Model")

metrics = ["Precision", "Recall", "F1-Score", "ROC-AUC", "PR-AUC"]

comparison = pd.concat({
    "Baseline": baseline[metrics],
    "Tuned": tuned[metrics]
}, axis=1)

comparison[("Improvement", "PR-AUC")] = tuned["PR-AUC"] - baseline["PR-AUC"]
comparison = comparison.sort_values(("Tuned", "PR-AUC"), ascending=False)

plot_confusion_matrices(
    trained_models, tuned_models, X_test, y_test,
    ranking=comparison.index,
)
display(comparison.round(3))


**Observation:** Tuning improved Gradient Boosting most (+0.13 PR-AUC), while XGBoost remains strongest overall (0.96 tuned PR-AUC). Tuning reduced PR-AUC for XGBoost and Random Forest, so retain the best-performing version per model and confirm choices through cross-validation.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Select Best Tuned Model Based on CV PR-AUC

Following model tuning, cross-validation PR-AUC was used as the primary criterion for selecting the final model. This approach evaluates how consistently each model identifies customers at risk of churn across different training-data splits. XGBoost was selected based on its strongest cross-validation performance, achieving a PR-AUC of 0.9481 (94.81%). The held-out test set was then used to assess final predictive performance.

In [ ]:
# Rank Tuned Models by Cross-Validation PR-AUC

cv_comparison = tuning_results[["CV PR-AUC"]].sort_values(
    "CV PR-AUC", ascending=False
)

display(cv_comparison.round(4))

# Select Best Model
best_model_name = cv_comparison.index[0]
best_model = tuned_models[best_model_name]

print(f"\nSelected Model: {best_model_name}")
print(f"CV PR-AUC: {cv_comparison.iloc[0, 0]:.4f}")

**Observation:** No direct target leakage was identified, but temporal leakage cannot be ruled out without observation and churn dates and timestamps (which wasnt available in the dataset). XGBoost was selected, achieving the highest cross-validation PR-AUC (0.95).

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Final Evaluation of Selected XGBoost Model

In [ ]:
final_results = evaluate_models(
    {best_model_name: best_model},
    X_train, X_test, y_train, y_test
)

display( final_results.set_index(["Model", "Dataset"]))


**Final test observation:** XGBoost correctly identified 357 of 379 churners (94% recall), missing 22. Of 417 accounts flagged as at risk, 357 churned (86% precision). Test accuracy was 96% and PR-AUC 0.96, indicating strong performance, with some unnecessary retention outreach.

---
### 5.1 Feature Importance and Model Explainability
XGBoost, with the highest cross-validation PR-AUC (0.95), was selected for feature importance analysis to identify the account characteristics most influential in its churn predictions.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### XGBoost Feature Importance
This reusable function extracts feature importance from the trained pipeline, combines one-hot encoded categorical features back into their original variables, and plots the results in descending order.

In [ ]:
# Aggregate and plot XGBoost importance by original feature.

def plot_feature_importance(model, top_n=17):
    """Plot top features and return their importance as percentages."""
    preprocessor = model.named_steps["preprocessor"]
    classifier = model.named_steps["classifier"]

    # Pair transformed feature names with their model importance scores.
    feature_df = pd.DataFrame({
        "Feature": preprocessor.get_feature_names_out(),
        "Importance": classifier.feature_importances_,
    })

    # Map encoded features back to their original input columns.
    original_cols = [
        col for _, _, cols in preprocessor.transformers_
        if isinstance(cols, (list, tuple, np.ndarray, pd.Index))
        for col in cols
    ]

    def original_feature(name):
        name = name.split("__", 1)[-1]
        return next(
            (col for col in sorted(original_cols, key=len, reverse=True)
             if name == col or name.startswith(col + "_")),
            name,
        )

    feature_df["Feature"] = feature_df["Feature"].map(original_feature)

    # Combine one-hot encoded feature scores under their source column.
    feature_df = (
        feature_df.groupby("Feature", as_index=False)["Importance"]
        .sum()
        .sort_values("Importance", ascending=False)
    )
    feature_df["Importance (%)"] = (
        100 * feature_df["Importance"] / feature_df["Importance"].sum()
    )

    # Display the top features in descending order from top to bottom.
    plot_df = feature_df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(plot_df["Feature"], plot_df["Importance (%)"])
    ax.bar_label(bars, fmt="%.2f%%", padding=4, fontsize=9)
    ax.set_xlim(0, plot_df["Importance (%)"].max() * 1.15)
    ax.set(xlabel="Relative Feature Importance (%)",
           ylabel="Account Features",
           title="XGBoost — Feature Importance")
    plt.tight_layout()
    plt.show()

    return feature_df

In [ ]:
# Plot Feature Importance
feature_importance = plot_feature_importance(best_model)

**Observations**

- Account segment (20.55%), payment (14.59%), and tenure (11.85%) are the top predictors.
- Marital status (8.25%) and complaints (6.98%) also contribute.
- Login device (6.61%) ranks above several numeric features, suggesting added predictive value.
- Other features each contribute less than 5%.

**Conclusion:** Feature importance broadly supports the EDA findings, especially for account segment, payment, and tenure. Importance indicates predictive contribution, not causation or effect direction.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### SHAP (Model Explainability)
XGBoost feature importance supports the EDA findings. SHAP analysis will show how individual features influence churn predictions and in which direction.

In [ ]:
# This function uses your selected XGBoost pipeline and its existing preprocessing steps.

def plot_shap_summary(model, X, sample_size=1000):
    X_sample = X.sample(n=min(sample_size, len(X)), random_state=42)
    preprocessor = model.named_steps["preprocessor"]
    classifier = model.named_steps["classifier"]

    X_transformed = preprocessor.transform(X_sample)
    if hasattr(X_transformed, "toarray"):
        X_transformed = X_transformed.toarray()

    feature_names = preprocessor.get_feature_names_out()
    explainer = shap.TreeExplainer(classifier)
    shap_values = explainer(X_transformed)

    shap.summary_plot(
        shap_values.values,
        X_transformed,
        feature_names=feature_names,
        max_display=20,
        plot_size=(14, 7),
        show=False
    )

    plt.title("XGBoost — SHAP Summary", pad=20)
    plt.tight_layout()
    plt.show()

    return shap_values

In [ ]:
# Run SHAP values
shap_values = plot_shap_summary(best_model, X_test)

**SHAP Observations**

- Tenure has the strongest influence: shorter tenure generally raises predicted churn risk.
- Recent complaints and customer contact are associated with higher predicted risk.
- Higher monthly revenue, agent scores, and account-user counts tend to increase predicted risk.
- Higher account tenure and the Super Plus segment generally lower predicted risk.

SHAP shows predictive associations, not causation.

---
###  6. Customer Segmentation

XGBoost was used to generate account-level churn risk scores for the held-out test set. Account IDs were retained to support targeted retention planning.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### 1. Generate Customer Churn Risk Scores

In [ ]:
# Generate account-level churn probabilities and risk segments.
def generate_churn_risk(model, X, accounts, y=None):
    """Return accounts ranked by predicted churn risk, with optional actual outcomes."""
    results = X[["account_segment", "tenure", "rev_per_month"]].copy()
    results.insert(0, "accountid", accounts.reindex(X.index))
    results["Churn Probability"] = model.predict_proba(X)[:, 1]

    # Group predicted probabilities into business-friendly risk bands.
    results["Risk Segment"] = pd.cut(
        results["Churn Probability"],
        bins=[-0.001, 0.30, 0.60, 0.80, 1.00],
        labels=["Low", "Moderate", "High", "Critical"],
    )

    if y is not None:
        results["Actual Churn"] = y.reindex(X.index)

    return results.sort_values("Churn Probability", ascending=False).reset_index(drop=True)

In [ ]:
# Score and rank the held-out test accounts.
churn_risk = generate_churn_risk(best_model, X_test, test_accounts, y_test)

risk_sample = pd.concat([
    churn_risk.groupby("Risk Segment", observed=True).sample(n=1, random_state=42),
    churn_risk.drop(
        churn_risk.groupby("Risk Segment", observed=True).sample(n=1, random_state=42).index
    ).sample(n=12, random_state=42)
]).sample(frac=1, random_state=42)

display(risk_sample.round(4))

**Observation:** 

This sample illustrates how XGBoost’s account-level risk scores may support retention prioritization: 2 of 4 High/Critical-risk accounts and 1 of 2 Moderate-risk accounts churned, while none of the 10 Low-risk accounts did. This is consistent with the model’s strong test performance (94% recall, 86% precision), though the small sample is illustrative, not a separate evaluation. ***Revenue exposure is based on historical `rev_per_month` and does not reflect future revenue, margins, or campaign costs.***

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

####  Summarize the Risk Distribution

In [ ]:
risk_summary = (
    churn_risk.groupby("Risk Segment", observed=True)
    .agg(
        Accounts=("accountid", "count"),
        Avg_Churn_Probability=("Churn Probability", "mean"),
        Avg_Monthly_Revenue=("rev_per_month", "mean")
    )
    .reindex(["Critical", "High", "Moderate", "Low"])
)

risk_summary["Accounts (%)"] = (
    risk_summary["Accounts"] / risk_summary["Accounts"].sum() * 100
)

display(risk_summary.round(2))

**Observations**

- Most accounts (78.24%) are low risk, while 372 (16.52%) are high or critical risk and should be prioritized for targeted retention.
- This is consistent with the previous account-level scoring and model evaluation, XGBoost distinguishes a smaller group with elevated predicted churn risk. Prioritize these accounts for retention assessment; revenue values reflect historical monthly averages in the dataset’s original units, not confirmed future revenue or campaign profitability.

**Next:** Assess revenue at risk and prioritize retention opportunities.


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Revenue-at-Risk Analysis

Revenue exposure is assessed using two measures: total monthly revenue by risk segment and probability-weighted revenue at risk, calculated as monthly revenue multiplied by predicted churn probability. The latter is an estimate, not a guaranteed loss.

In [ ]:
# Summarize monthly revenue exposure by churn-risk segment.

def analyze_revenue_risk(churn_risk):
    """Report revenue exposure and account shares by risk segment."""
    df = churn_risk.copy()
    risk_col = "Probability-Weighted Monthly Revenue at Risk"
    df[risk_col] = df["rev_per_month"] * df["Churn Probability"]

    # Aggregate account counts and revenue measures by segment.
    summary = (
        df.groupby("Risk Segment", observed=True)
        .agg(
            Accounts=("accountid", "count"),
            **{
                "Monthly Revenue": ("rev_per_month", "sum"),
                risk_col: (risk_col, "sum"),
                "Average Churn Probability": ("Churn Probability", "mean"),
            },
        )
        .reindex(["Critical", "High", "Moderate", "Low"])
    )

    # Calculate each segment's share of accounts and estimated revenue at risk.
    summary["Accounts (%)"] = summary["Accounts"] / len(df) * 100
    summary["Share of Estimated Revenue at Risk (%)"] = (
        summary[risk_col] / summary[risk_col].sum() * 100
    )

    # Add combined High and Critical figures for prioritization.
    priority = df[df["Risk Segment"].isin(["Critical", "High"])]
    summary.loc["High + Critical"] = {
        "Accounts": len(priority),
        "Monthly Revenue": priority["rev_per_month"].sum(),
        risk_col: priority[risk_col].sum(),
        "Average Churn Probability": priority["Churn Probability"].mean(),
        "Accounts (%)": len(priority) / len(df) * 100,
        "Share of Estimated Revenue at Risk (%)": (
            priority[risk_col].sum() / df[risk_col].sum() * 100
        ),
    }

    display(summary.round(2))
    return summary

In [ ]:
# Revenue at Risk Summary
revenue_risk_summary = analyze_revenue_risk(churn_risk)

**Revenue-at-Risk Observation and Business Impact:**

The analysis identified 372 High- and Critical-risk accounts, representing only 16.52% of the 2,252 evaluated accounts. However, these accounts contribute 78.60% of estimated probability-weighted monthly revenue exposure (2,525.38 in the dataset's original units).

Critical-risk accounts alone account for 68.36% of total estimated exposure.

**Business Implication:** Rather than distributing retention resources equally across all customers, the business should prioritize these 372 accounts, starting with those having the greatest estimated revenue exposure. This targeted approach could improve resource allocation, reduce unnecessary campaign spending and increase the potential financial benefit of retention activities.

These figures represent model-based estimates of potential revenue exposure, not confirmed revenue losses or realized savings.

---
### 7. Retention Strategy

Section 6 identified the High- and Critical-risk accounts with the greatest estimated revenue exposure. This section translates those findings into targeted retention actions and an account-prioritization approach.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Define Retention Strategies

In [ ]:
# Define retention actions.
retention_actions = {
    "Critical": "Immediate personalized outreach, complaint resolution and tailored retention incentives",
    "High": "Proactive engagement, targeted retention offers and loyalty incentives",
    "Moderate": "Preventive engagement and service improvement",
    "Low": "Routine monitoring and customer engagement"
}

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Prioritize Accounts for Retention

The function calculates each account’s probability-weighted monthly revenue exposure and ranks accounts by risk segment and exposure to support retention prioritization.

In [ ]:
# Rank accounts by churn risk and estimated monthly revenue exposure.
def prioritize_accounts_for_retention(churn_risk):
    """Prioritize accounts by risk segment and probability-weighted revenue."""
    df = churn_risk.copy()

    df["Probability-Weighted Monthly Revenue at Risk"] = (
        df["rev_per_month"] * df["Churn Probability"]
    )

    risk_order = {"Critical": 0, "High": 1, "Moderate": 2, "Low": 3}
    df["Retention Priority"] = df["Risk Segment"].astype(str).map(risk_order) + 1
    df["Recommended Action"] = df["Risk Segment"].astype(str).map(retention_actions)

    return (
        df.sort_values(
            ["Retention Priority", "Probability-Weighted Monthly Revenue at Risk"],
            ascending=[True, False]
        )
        .reset_index(drop=True)
    )

# Generate and review prioritized accounts.
retention_plan = prioritize_accounts_for_retention(churn_risk)
display(retention_plan.head(20).round(2))

**Observation:** The 20 highest-priority accounts are all classified as Critical risk; 19 had churned in the held-out test data. Ranking by probability-weighted monthly revenue also surfaces high-revenue accounts, such as account 24998 (estimated exposure: 125.77 in dataset units).

This is a retrospective view of test accounts, not a live campaign list. The results illustrate how risk and revenue exposure can guide prioritization, but do not establish campaign effectiveness.

**Business Implications**
- Prioritize high-risk accounts with greater estimated revenue exposure.
- Use complaints and short tenure to tailor customer support.
- Validate retention actions by comparing outcomes across treatment and control groups.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Illustrative Retention Offer ROI & Break-even Analysis

Illustrative ROI and break-even costs for Critical- and High-risk accounts are estimated using probability-weighted revenue at risk, assumed retention success rates and offer-based campaign costs. **Break-even cost per account** represents the maximum affordable intervention spending before net benefit becomes negative. Sensitivity analysis assesses financial viability across Conservative, Base Case and Optimistic scenarios, with actual returns requiring pilot validation.

##### Define Campaign Assumptions
The estimates below are illustrative, not observed campaign results; the 372 accounts come from the held-out test set.

In [ ]:
# Define campaign assumptions.

retention_success_scenarios={
    "Conservative":{"Critical":0.15,"High":0.20},
    "Base Case":{"Critical":0.30,"High":0.40},
    "Optimistic":{"Critical":0.45,"High":0.60}
}

retention_offers={
    "Personalized Outreach":0.05,
    "Loyalty Reward":0.10,
    "Discount Voucher":0.15,
    "Cashback Incentive":0.25,
    "Premium Benefit":0.60
}

##### Evaluate illustrative retention offer ROI and Break-Even Costs.

In [ ]:
# Evaluate illustrative retention-offer ROI and break-even costs.

def evaluate_campaign_impact(retention_plan, months=12, margin=0.30):
    """Calculate illustrative ROI across three retention scenarios."""

    # Evaluate only the Critical- and High-risk accounts.
    df = retention_plan[retention_plan["Risk Segment"].isin(["Critical", "High"])]
    results = []

    # Compare each assumed retention-success scenario by risk segment.
    for scenario, rates in retention_success_scenarios.items():
        for segment, group in df.groupby("Risk Segment", observed=True):
            accounts = len(group)
            revenue = group["rev_per_month"].sum()

            # Estimate contribution from probability-weighted revenue exposure.
            contribution = (group["Probability-Weighted Monthly Revenue at Risk"].sum()    * rates[segment]     * months    * margin  )

            # Calculate cost, net benefit, and ROI for each offer.
            for offer, pct in retention_offers.items():
                cost = revenue * pct
                net = contribution - cost
                results.append([scenario, segment, offer, pct * 100, rates[segment] * 100, accounts, cost, contribution / accounts, net,  net / cost * 100, net > 0,])

    # Assign descriptive column names to the scenario results.
    columns = [
        "Scenario", "Risk Segment", "Retention Offer", "Offer (%)","Success Rate (%)", "Accounts", "Estimated Campaign Cost",
        "Break-even Cost/Account", "Estimated Net Benefit", "Illustrative ROI (%)", "Viable"]

    # Return the results rounded to two decimal places.
    return pd.DataFrame(results, columns=columns).round(2)


In [ ]:
campaign_impact=evaluate_campaign_impact(retention_plan)
display(campaign_impact)

In [ ]:
# ROI sensitivity comparison

roi_sensitivity=campaign_impact.pivot(
    index=["Risk Segment","Retention Offer","Offer (%)"],
    columns="Scenario",
    values="Illustrative ROI (%)"
).reset_index()

roi_sensitivity.columns.name=None
display(roi_sensitivity)

In [ ]:
# Top 5 segment-offer combinations by highest ROI
top_5=(campaign_impact
    .sort_values("Illustrative ROI (%)",ascending=False)
    .drop_duplicates(["Risk Segment","Retention Offer"])
    .head(5)
    .reset_index(drop=True))

display(top_5)

**Key Observations — Illustrative ROI Sensitivity Analysis**

- **Best-performing offer:** Personalized Outreach (5%) consistently delivers the highest illustrative ROI across all three scenarios.
- **Top 5 combinations:** Under the Optimistic scenario, Personalized Outreach achieves **2,952.78% ROI for Critical** and **2,886.60% for High**, followed by Loyalty Rewards (10%) and Critical-segment Discount Vouchers (15%).
- **Sensitivity analysis:** Outreach, Loyalty Rewards, Discount Vouchers and Cashback Incentives remain financially viable across all three scenarios.
- **Higher-cost interventions:** Premium Benefits (60%) produce negative ROI under Conservative assumptions (**−15.20% Critical; −17.04% High**).
- **Recommendation:** Pilot **Personalized Outreach (5%)**, **Loyalty Rewards (10%)** and **Discount Vouchers (15%)**, prioritizing cost-effective interventions before expensive incentives.

*Note: Top 5 results reflect Optimistic success assumptions (45% Critical; 60% High), not guaranteed outcomes. Actual ROI requires validation against control groups.*

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Campaign Validation and Monitoring

Pilot retention offers with treatment and control groups to validate effectiveness and financial returns.

| KPI | Purpose |
|---|---|
| Retention uplift | Measure effectiveness against control |
| Cost per account | Compare with break-even cost |
| Net benefit | Assess profitability |
| ROI by offer and risk segment | Identify the best-performing offers |

**Goal:** Validate illustrative ROI before wider rollout.


---
## <span style="color:#1976D2;">Section 4: Streamlit Application and Cloud Deployment</span>

Following the successful development and evaluation of the XGBoost model, ChurnGuard was implemented as a cloud-hosted Prediction App using Streamlit. The application enables users to generate churn predictions for individual accounts or uploaded datasets without retraining the model. It also provides risk classifications, recommended retention actions, downloadable prediction results and automated Gmail alerts for newly scored High- and Critical-risk accounts.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 1. Save the Selected Best Model

The existing `best_model` is a fitted scikit-learn pipeline containing both preprocessing and the XGBoost classifier. Saving the complete pipeline ensures that new customer data receives the same preprocessing used during training.

In [ ]:
# Save the best model and its input schema for Streamlit.
model_file=f"churnguard_{best_model_name.lower().replace(' ','_')}.joblib"
joblib.dump(best_model,MODELS_DIR/model_file)

numeric=X_train.select_dtypes(include="number").columns.tolist()
categorical=X_train.select_dtypes(exclude="number").columns.tolist()

# Preserve feature order, default values and dropdown options.
schema={
    "model_file":model_file,
    "features":X_train.columns.tolist(),
    "numeric":numeric,
    "defaults":X_train[numeric].median().fillna(0).to_dict(),
    "categories":{c:X_train[c].dropna().astype(str).unique().tolist() for c in categorical}
}

(MODELS_DIR/"feature_schema.json").write_text(json.dumps(schema,indent=2))
print(f"Saved: {model_file} and feature_schema.json")

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 2. Dependency Management and Streamlit Application

The `requirements.txt` file defines the application dependencies, while `app.py` loads the saved model, validates input features, and generates churn probabilities, risk segments and retention recommendations.

In [ ]:
# Generate Streamlit dependencies in the current project folder
!pip freeze | grep -E "^(streamlit|pandas|numpy|scikit-learn|xgboost|joblib|openpyxl)==" > requirements.txt
!cat requirements.txt


#### Streamlit App

In [ ]:
%%writefile app.py
import io,json,re,sqlite3,hashlib,smtplib,ssl
from email.message import EmailMessage
from pathlib import Path
import joblib,pandas as pd,streamlit as st

# App settings and branding
ROOT=Path(__file__).resolve().parent
st.set_page_config(page_title='ChurnGuard',page_icon='🛡️',layout='wide')
st.sidebar.title('🛡️ ChurnGuard')
st.sidebar.caption('Customer Churn Prediction')

# Valid training ranges and categories
BOUNDS={'tenure':(0,99),'city_tier':(1,3),'cc_contacted_ly':(4,132),'service_score':(0,5),'account_user_count':(1,6),'cc_agent_score':(1,5),'rev_per_month':(1,140),'complain_ly':(0,1),'rev_growth_yoy':(4,28),'coupon_used_for_payment':(0,16),'day_since_cc_connect':(0,47),'cashback':(0,1997)}
CATEGORIES={'payment':['cash on delivery','credit card','debit card','e wallet','upi'],'gender':['female','male'],'account_segment':['hni','regular','regular plus','super','super plus'],'marital_status':['divorced','married','single'],'login_device':['computer','mobile','others']}
ACTIONS={'Critical':'Immediate personalized outreach, complaint resolution and tailored incentives','High':'Proactive engagement, targeted retention offers and loyalty incentives','Moderate':'Preventive engagement and service improvement','Low':'Routine monitoring and customer engagement'}

# Load model and feature schema once
@st.cache_resource
def load_assets():
    schema=json.loads((ROOT/'models/feature_schema.json').read_text(encoding='utf-8'))
    return joblib.load(ROOT/'models'/schema['model_file']),schema

try: model,schema=load_assets()
except Exception as e:
    st.error(f'Unable to load model/schema: {e}')
    st.stop()
features=schema['features']

# Normalize incoming column names
def normalize(df):
    df=df.copy()
    df.columns=[re.sub(r'[\s-]+','_',str(c).strip().lower()) for c in df.columns]
    if df.columns.duplicated().any(): raise ValueError('Duplicate column names after normalization.')
    return df

# Read real CSV or XLSX content, even with a misleading extension
def read_table(upload):
    raw=upload.getvalue()
    if not raw: raise ValueError('The uploaded file is empty.')
    if raw.startswith(b'PK\x03\x04'):
        return pd.read_excel(io.BytesIO(raw),engine='openpyxl',dtype={'accountid':'string'})
    if raw.startswith(b'\xd0\xcf\x11\xe0'):
        raise ValueError('Old Excel (.xls) format detected. Save as .xlsx and upload again.')
    for encoding in ('utf-8-sig','utf-16','cp1252','latin-1'):
        try:
            df=pd.read_csv(io.StringIO(raw.decode(encoding)),sep=None,engine='python',dtype={'accountid':'string'})
            if len(df.columns)>1: return df
        except (UnicodeError,pd.errors.ParserError,ValueError): pass
    raise ValueError('Cannot read the file. Upload a customer CSV or XLSX with the required features.')

# Validate features, calculate probabilities and assign retention actions
def score(df):
    df=normalize(df)
    if df.empty: raise ValueError('No customer records found.')
    # Account ID is required for reliable tracking and email deduplication
    if 'accountid' not in df: raise ValueError('Missing accountid column. Every customer needs an Account ID.')
    df['accountid']=df['accountid'].astype('string').str.strip()
    if df['accountid'].isna().any() or df['accountid'].eq('').any(): raise ValueError('Account ID cannot be blank.')
    if df['accountid'].duplicated().any(): raise ValueError('Duplicate account IDs found. Include one row per customer.')
    missing=[f for f in features if f not in df]
    if missing: raise ValueError('Missing model features: '+', '.join(missing))
    x=df[features].copy()
    for f in schema['numeric']:
        x[f]=pd.to_numeric(x[f],errors='coerce')
        if f in BOUNDS:
            lo,hi=BOUNDS[f]
            if (~x[f].dropna().between(lo,hi)).any(): raise ValueError(f'{f} must be between {lo} and {hi}.')
    for f in schema['categories']:
        x[f]=x[f].astype('string').str.strip().str.lower().astype(object)
        if f in CATEGORIES:
            invalid=x[f].dropna()[~x[f].dropna().isin(CATEGORIES[f])].unique().tolist()
            if invalid: raise ValueError(f'{f} has unsupported values: {invalid}')
    probability=model.predict_proba(x)[:,1]
    risk=pd.cut(probability,[-.001,.30,.60,.80,1],labels=['Low','Moderate','High','Critical']).astype(str)
    result=pd.DataFrame({'accountid':df['accountid'].to_numpy()})
    for f in ['account_segment','payment','tenure','marital_status','complain_ly','login_device']:
        if f in x: result[f]=x[f].to_numpy()
    result['Churn Probability']=probability
    result['Risk Segment']=risk
    if 'rev_per_month' in x: result['Revenue at Risk']=x['rev_per_month'].to_numpy()*probability
    result['Recommended Action']=[ACTIONS[r] for r in risk]
    return result.sort_values('Churn Probability',ascending=False).reset_index(drop=True)

# Show predictions with downloadable CSV and XLSX
def show_results(result,name):
    st.dataframe(result,hide_index=True,width="stretch")
    csv_col,xlsx_col=st.columns(2)
    csv_col.download_button('Download CSV',result.to_csv(index=False).encode('utf-8-sig'),f'{name}.csv','text/csv')
    output=io.BytesIO()
    with pd.ExcelWriter(output,engine='openpyxl') as writer: result.to_excel(writer,index=False,sheet_name='Predictions')
    xlsx_col.download_button('Download Excel',output.getvalue(),f'{name}.xlsx','application/vnd.openxmlformats-officedocument.spreadsheetml.sheet')

# Email high-risk alerts; SQLite prevents duplicate messages across app reruns
DB=ROOT/'churnguard_alerts.sqlite3'
def send_alerts(result):
    alerts=result[result['Risk Segment'].isin(['Critical','High'])].copy()
    if alerts.empty: return 'No Critical or High-risk accounts to notify.'
    try:
        cfg=st.secrets['email']
        host,port,user,password=cfg['host'],int(cfg.get('port',587)),cfg['username'],cfg['password']
        sender=cfg.get('sender',user)
        recipients=cfg['recipients']
        recipients=[v.strip() for v in recipients.split(',')] if isinstance(recipients,str) else list(recipients)
        if not recipients: raise ValueError('No email recipients configured.')
    except Exception as e: raise ValueError('Configure [email] in .streamlit/secrets.toml (host, port, username, password, recipients).') from e
    with sqlite3.connect(DB,timeout=15) as con:
        con.execute('CREATE TABLE IF NOT EXISTS sent (account TEXT PRIMARY KEY, risk TEXT, sent_at TEXT DEFAULT CURRENT_TIMESTAMP)')
        # Alert once per account and risk level; a change in level generates a new alert
        pending=[]
        for _,r in alerts.iterrows():
            account=str(r['accountid']).strip()
            if not account or account.lower() in ('n/a','nan','none'): continue
            key=hashlib.sha256((account+'|'+str(r['Risk Segment'])).encode()).hexdigest()
            if not con.execute('SELECT 1 FROM sent WHERE account=?',(key,)).fetchone(): pending.append((key,r))
        if not pending: return 'No new alerts; previously notified accounts were skipped.'
        lines=['ChurnGuard early warning — new high-risk accounts','','Account ID | Risk | Churn Probability | Recommended Action']
        for _,r in pending:
            lines.append(f"{r['accountid']} | {r['Risk Segment']} | {r['Churn Probability']:.1%} | {r['Recommended Action']}")
        msg=EmailMessage()
        msg['Subject']=f'ChurnGuard: {len(pending)} new high-risk account(s)'
        msg['From']=sender
        msg['To']=', '.join(recipients)
        msg.set_content('\n'.join(lines))
        if port==465:
            with smtplib.SMTP_SSL(host,port,context=ssl.create_default_context(),timeout=30) as smtp:
                smtp.login(user,password); smtp.send_message(msg)
        else:
            with smtplib.SMTP(host,port,timeout=30) as smtp:
                smtp.starttls(context=ssl.create_default_context())
                smtp.login(user,password); smtp.send_message(msg)
        con.executemany('INSERT INTO sent(account,risk) VALUES (?,?)',[(key,str(r['Risk Segment'])) for key,r in pending])
        con.commit()
    return f'Email sent for {len(pending)} new Critical/High-risk account(s).'

# Automatic email after scoring; suppress repeat attempts on Streamlit reruns
# Persistent SQLite history prevents resending already notified account/risk pairs.
def auto_alert(result,run_id):
    if not result['Risk Segment'].isin(['Critical','High']).any(): return
    if st.session_state.get('last_alert_run')==run_id: return
    st.session_state['last_alert_run']=run_id
    try:
        message=send_alerts(result)
        if message.startswith('Email sent'):
            st.success('Email alerts: '+message)
        else:
            st.info(message)
    except Exception as e: st.warning(f'Automatic email alert failed: {e}')

# Sidebar navigation
page=st.sidebar.radio('Navigation',['Single Prediction','Batch Prediction','Model Information'])
st.title('🛡️ ChurnGuard')
st.caption('Customer Churn Prediction and Retention Strategy')

if page=='Single Prediction':
    st.subheader('Single Account Prediction')
    with st.form('single'):
        accountid=st.text_input('Account ID',help='Required: unique identifier for the customer')
        values={}
        with st.expander('Numerical Features',expanded=True):
            cols=st.columns(3)
            for i,f in enumerate(schema['numeric']):
                with cols[i%3]:
                    default=float(schema.get('defaults',{}).get(f,0) or 0)
                    if f in BOUNDS:
                        lo,hi=BOUNDS[f]
                        values[f]=st.number_input(f,min_value=float(lo),max_value=float(hi),value=max(float(lo),min(default,float(hi))))
                    else: values[f]=st.number_input(f,value=default)
        with st.expander('Categorical Features',expanded=True):
            cols=st.columns(3)
            for i,f in enumerate(schema['categories']):
                with cols[i%3]:
                    options=CATEGORIES.get(f) or schema['categories'].get(f) or ['']
                    values[f]=st.selectbox(f,options)
        submitted=st.form_submit_button('Predict Churn',type='primary')
    if submitted:
        try:
            result=score(pd.DataFrame([{**values,'accountid':accountid}]))
            a,b=st.columns(2)
            a.metric('Churn Probability',f"{result['Churn Probability'].iloc[0]:.1%}")
            b.metric('Risk Segment',result['Risk Segment'].iloc[0])
            st.info(result['Recommended Action'].iloc[0])
            show_results(result,'single_prediction')
            auto_alert(result,'single:'+hashlib.sha256(json.dumps({'accountid':accountid,**values},sort_keys=True,default=str).encode()).hexdigest())
        except Exception as e: st.error(str(e))
elif page=='Batch Prediction':
    st.subheader('Batch Account Prediction')
    upload=st.file_uploader('Upload customer CSV or XLSX',type=['csv','xlsx'])
    st.caption('Include a unique, nonblank accountid for each customer and all required features: '+', '.join(features))
    st.caption('Upload customer records, not a feature-importance report.')
    if upload:
        try:
            df=read_table(upload)
            result=score(df)
            a,b,c=st.columns(3)
            a.metric('Accounts',len(result))
            b.metric('Critical Risk',int(result['Risk Segment'].eq('Critical').sum()))
            c.metric('High Risk',int(result['Risk Segment'].eq('High').sum()))
            show_results(result,'batch_predictions')
            auto_alert(result,'batch:'+hashlib.sha256(upload.getvalue()).hexdigest())
        except Exception as e: st.error(str(e))
else:
    st.subheader('Model Information')
    name=Path(schema['model_file']).stem.removeprefix('churnguard_').replace('_',' ').title()
    st.write('**Selected Model:**',name)
    st.write('**Model Type:** Supervised binary classification')
    st.write('**Target:** Churn')
    st.write('**Predictors:**',len(features))
    st.write('**Preprocessing:** Included in saved pipeline')
    st.table(pd.DataFrame({'Risk Segment':['Low','Moderate','High','Critical'],'Churn Probability':['0–30%','>30–60%','>60–80%','>80–100%']}))
    st.caption('Previously reported evaluation: CV PR-AUC 0.9481 | Test PR-AUC 0.96 | Accuracy 96%')
st.caption('Predictions are estimates; campaign ROI requires pilot validation.')


#### Email Smtp Configuration

In [ ]:
# ChurnGuard Gmail settings
email_config={
    "host":os.getenv("host","smtp.gmail.com"),
    "port":int(os.getenv("port","587")),
    "username":os.getenv("username") or input("Gmail address: "),
    "password":os.getenv("password") or getpass("Gmail App Password: "),
    "sender":os.getenv("sender") or None,
    "recipients":os.getenv("recipients") or ""
}

email_config["sender"]=email_config["sender"] or email_config["username"]
email_config["recipients"]=[x.strip() for x in email_config["recipients"].split(",") if x.strip()]


<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 3. Optional Local Test

In [ ]:
# !streamlit run app.py

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 4. Push Deployment Files

```mk
# Add, commit, and push changes to GitHub
    # Initialize Git and push the complete ChurnGuard project
    !git init
    !git branch -M main
    !git status
    !git add .
    !git commit -m "Initial commit: ChurnGuard ML project and Streamlit deployment"
    !git remote add origin https://github.com/****/ChurnGuard.git
    !git push -u origin main

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### 5. Streamlit Space

**Streamlit Space:** https://predict-customerchurn.streamlit.app/


### Streamlit Prediction Application
<div style="display: flex; gap: 15px;">
    <img src="app1.png" width="50%" height="400">
    <img src="app2.png" width="50%" height="400">
</div>

---
## <span style="color:#1976D2;">Final Business Recommendations and Project Conclusion</span>

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

#### Business Recommendations

- **Prioritize high-risk customers:** Target the 372 Critical- and High-risk accounts, representing **78.60% of estimated revenue exposure**, prioritizing accounts with the highest revenue at risk.
- **Implement targeted retention:** Pilot Personalized Outreach (5%), Loyalty Rewards (10%) and Discount Vouchers (15%), prioritizing cost-effective interventions.
- **Automate early warnings:** Use the ChurnGuard Prediction App to identify high-risk accounts and send automatic email alerts to support timely intervention.
- **Maintain preventive engagement:** Monitor the 118 Moderate-risk and 1,762 Low-risk accounts, escalating interventions as risk increases.
- **Control campaign costs:** Restrict Premium Benefits (60%) to exceptional high-value accounts due to negative ROI under Conservative assumptions.
- **Validate before scaling:** Evaluate retention uplift, campaign costs and actual ROI through controlled pilots.

*Note: ROI estimates are illustrative and based on assumed retention success rates and costs.*

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### Project Conclusion

ChurnGuard successfully developed an **XGBoost churn prediction model**, achieving **96% accuracy, 94% recall, 0.90 F1-score, 0.96 PR-AUC and 0.9481 cross-validation PR-AUC**.

Account segment (20.55%), payment method (14.59%) and tenure (11.85%) emerged as the leading predictive factors. Risk segmentation and ROI analysis further support targeted retention decisions.

Beyond prediction, the project demonstrates how risk segmentation, estimated revenue exposure and illustrative ROI analysis can support targeted retention decisions. The ChurnGuard Prediction App has been implemented to provide individual and batch churn predictions, risk classification, recommended retention actions, downloadable results and automated Gmail alerts for newly scored High- and Critical-risk accounts. These capabilities provide a practical foundation for proactive retention management, while actual campaign effectiveness and financial returns remain subject to validation.

<hr style="border: 0; border-top: 0.5px solid #2b5390;">

### Limitations and Future Enhancements

The dataset lacks churn timestamps, observed campaign outcomes, verified intervention costs, contribution margins and defined revenue units. These limitations prevent temporal leakage from being ruled out and actual campaign ROI from being established.

**Implemented:** The ChurnGuard Prediction App supports individual and batch predictions, churn probability scoring, risk classification, personalized retention recommendations, downloadable results and automated Gmail alerts for newly scored High- and Critical-risk accounts.

**Future Enhancements:** Introduce account-level tracking to monitor changes in churn risk, retention interventions and customer outcomes over time. Additional improvements include scheduled prediction updates, CRM integration, model calibration and performance monitoring.

**Next Steps:** Conduct controlled retention campaigns to measure actual retention improvements, validate financial assumptions and determine the most cost-effective interventions before wider implementation.

---